# FinSafe 03 — finance baseline and five-step training smoke test
Run after notebook 02, in the same Kaggle session. Keep GPU, Internet and HF_TOKEN
enabled. Uses GPU 0 only; the second T4 is not pooled with its memory.

Every code cell logs timestamps, duration and failures. Generation resumes from
saved responses; training saves full-state checkpoints every optimizer step.
The notebook performs only **five optimizer steps**, not the long research run.
Download the recovery ZIP before ending your session. Real safety evaluation
still needs an independent, reviewed benchmark.


In [ ]:
import sys, os, json, subprocess, time, datetime
from pathlib import Path
started=time.monotonic()
print(f"[{datetime.datetime.now(datetime.timezone.utc).isoformat()}] START stage03 bootstrap",flush=True)
try:
    project=Path("/kaggle/working/finsafe")
    if not project.exists():
        raise RuntimeError("Restore the project/data or run notebook 01 and 02 first")
    status=subprocess.check_output(["git","-C",str(project),"status","--porcelain","--untracked-files=no"],text=True)
    if status.strip():
        raise RuntimeError("Save tracked-file edits before updating")
    subprocess.run(["git","-C",str(project),"pull","--ff-only"],check=True)
    sys.path.insert(0,str(project/"src"))
    from finsafe.progress import Progress
    progress=Progress(project/"runs/notebook-03/logs","notebook-03")
    progress.log(f"DONE bootstrap in {time.monotonic()-started:.1f}s")
except BaseException as error:
    print(f"FAILED bootstrap after {time.monotonic()-started:.1f}s ({type(error).__name__})",flush=True)
    raise


In [ ]:
with progress.stage("01-prerequisites"):
    from kaggle_secrets import UserSecretsClient
    from finsafe.data import read_jsonl
    runtime_env=os.environ.copy()
    runtime_env.update(HF_TOKEN=UserSecretsClient().get_secret("HF_TOKEN"),
        HF_HOME="/kaggle/working/.cache/huggingface",HF_NAMESPACE="Finsafe-team",
        CUDA_VISIBLE_DEVICES="0",TOKENIZERS_PARALLELISM="false",TRL_ONLY_MODE="1",
        DISABLE_UNSLOTH_FOR_TRL="1",PYTHONUNBUFFERED="1")
    gpu_py=str(project/".venv-gpu/bin/python")
    cli=str(project/".venv-gpu/bin/finsafe")
    revisions=json.loads((project/"runs/environment/model-revisions.json").read_text())
    global_sha=revisions["CohereLabs/tiny-aya-global"]
    filtered=project/"data/processed/tatqa-tokenized"
    curated=filtered/"curated"
    train_rows=read_jsonl(curated/"sft_messages.jsonl")
    if len(train_rows)<40:
        raise RuntimeError("Too few accepted training examples for the 5-step, effective-batch-8 smoke test")
    progress.log(f"Accepted curated training examples: {len(train_rows)}")
    progress.log((filtered/"token_manifest.json").read_text())
    progress.run([cli,"doctor","--gpu"],cwd=project,env=runtime_env)


## Preserve the full finance validation baseline
This generates the original model on all accepted validation questions, not the
final test. Expect several minutes. Earlier smoke outputs remain unchanged.
Every response is checkpointed. This cell can be rerun after an interruption.


In [ ]:
with progress.stage("02-full-finance-validation-baseline"):
    validation=filtered/"validation.jsonl"
    baseline_out=project/"runs/baseline-validation-4bit/predictions.jsonl"
    progress.run([gpu_py,"-u","scripts/generate.py","--model","CohereLabs/tiny-aya-global",
        "--revision",global_sha,"--data",str(validation),"--condition","aligned-4bit",
        "--out",str(baseline_out),"--max-new-tokens","64","--load-in-4bit","--resume"],cwd=project,env=runtime_env)
    progress.run([cli,"summarize","--data",str(validation),"--predictions",str(baseline_out),
        "--out","runs/baseline-validation-4bit/report.json"],cwd=project,env=runtime_env)
    progress.run([cli,"audit-replay","--data",str(validation),"--predictions",str(baseline_out),
        "--out","runs/baseline-validation-4bit/auditkit.json"],cwd=project,env=runtime_env)


## Five optimizer steps, with full recovery state saved every step
Batch size 1, gradient accumulation 8, LoRA rank 16, max sequence 1024, NF4 with
FP16 compute. Check finite loss, trainable adapter parameter count and memory.
At most two recent checkpoints are retained. This is a compatibility test.
If interrupted, rerun this cell unchanged: --resume restores the latest complete
checkpoint. It may repeat work since the last completed checkpoint.


In [ ]:
with progress.stage("03-five-step-training"):
    smoke_dir=project/"checkpoints/smoke-sft-five"
    progress.run([gpu_py,"-u","scripts/train_sft.py","--revision",global_sha,
        "--data",str(curated),"--out",str(smoke_dir),"--max-steps","5",
        "--save-steps","1","--load-in-4bit","--resume"],cwd=project,env=runtime_env)


In [ ]:
with progress.stage("04-check-training-artifacts"):
    import math
    complete=json.loads((smoke_dir/"completed.json").read_text())
    if complete["global_step"]!=5:
        raise RuntimeError("Training did not reach five optimizer steps")
    loss=complete.get("metrics",{}).get("train_loss")
    if loss is None or not math.isfinite(float(loss)):
        raise RuntimeError("Training loss is missing or nonfinite")
    if not (smoke_dir/"adapter_config.json").is_file() or not (smoke_dir/"adapter_model.safetensors").is_file():
        raise RuntimeError("Expected saved LoRA adapter files not found")
    checkpoint=smoke_dir/"checkpoint-5"
    for required in ["trainer_state.json","optimizer.pt","scheduler.pt"]:
        if not (checkpoint/required).is_file():
            raise RuntimeError(f"Full recovery state missing: {required}")
    if not list(checkpoint.glob("rng_state*.pth")):
        raise RuntimeError("RNG recovery state missing")
    progress.log(f"CHECKPOINT verified at step 5; train_loss={loss}")
    progress.log(json.dumps(complete,indent=2))


## Adapter generation check on the same eight examples
Use the original aligned revision and the saved LoRA. This does not select the
final method or establish meaningful gains after just five steps.


In [ ]:
with progress.stage("05-adapter-smoke-generation"):
    smoke_data=project/"data/processed/tatqa-smoke.jsonl"
    sft_out=project/"runs/sft-five-smoke/predictions.jsonl"
    progress.run([gpu_py,"-u","scripts/generate.py","--model","CohereLabs/tiny-aya-global",
        "--revision",global_sha,"--adapter",str(smoke_dir),"--data",str(smoke_data),
        "--condition","finance-five-step-smoke","--out",str(sft_out),
        "--max-new-tokens","64","--load-in-4bit","--resume"],cwd=project,env=runtime_env)
    progress.run([cli,"summarize","--data",str(smoke_data),"--predictions",str(sft_out),
        "--out","runs/sft-five-smoke/report.json"],cwd=project,env=runtime_env)
    for row in read_jsonl(sft_out)[:3]:
        progress.log(f"Adapter response {row['id']}: {row['output']}")


## Download recovery archive
This includes full training checkpoints, prepared data and logs. It may be larger
than the stage-02 archive. It excludes token stores, caches and environments.
Download it or retain Kaggle output; local storage alone does not survive every
session reset. It does not automatically upload anything to Hugging Face.


In [ ]:
with progress.stage("06-recovery-backup"):
    import zipfile
    from IPython.display import FileLink,display
    backup=Path("/kaggle/working/finsafe-stage03-recovery.zip")
    with zipfile.ZipFile(backup,"w",compression=zipfile.ZIP_DEFLATED) as archive:
        for folder in [project/"data/processed",project/"runs",smoke_dir]:
            for file in folder.rglob("*"):
                if file.is_file() and not file.is_symlink() and not file.name.startswith(".env"):
                    archive.write(file,str(file.relative_to(project)))
    progress.log(f"Recovery ZIP: {backup.name} ({backup.stat().st_size/2**20:.1f} MiB)")
    display(FileLink(str(backup)))
    progress.log("Stage 03 complete. Share checkpoint verification, full baseline report and training loss before a longer run.")
